# vicuna-13b-local — dbbench-std 300문제 전체 실행 (Colab, Docker 없이)

**Colab Pro 권장** (런타임 유형: GPU, 가능하면 A100/L4 — 13B 4비트 양자화 + 300문제 x 최대 15라운드는
무료 T4로는 매우 오래 걸릴 수 있습니다).

이 노트북은 `project_db`의 공식 4개 모델(`gpt-3.5-turbo`/`gpt-4o`/`llama-3.1-8b`/`claude-sonnet-5`)이
쓴 것과 **똑같은 THUDM/AgentBench의 dbbench 채점 로직을 그대로 이식**해서, Docker 없이(Colab이
일반적으로 Docker를 지원하지 않으므로) MySQL을 직접 설치해 돌립니다. 아래 항목을 원본
`AgentBench/src/server/tasks/dbbench/{__init__.py,Interaction.py}`에서 그대로 가져왔습니다:

- `big_prompt` (시스템 프롬프트) — 완전히 동일한 문구
- `build_init_sql()` — 문제마다 테이블 생성 + 실제 행 데이터 삽입
- `Action: (.*?)\n` / ```` ```sql\n([\s\S]*?)\n``` ```` / `Final Answer:` 파싱 정규식 — 완전히 동일
- SELECT류는 정답 텍스트 비교, INSERT/UPDATE는 MD5 해시 비교 — 완전히 동일한 채점 쿼리
- 상태 문자열(`completed`/`agent validation failed`/`task limit reached`/`agent context limit`/`unknown`) — `src/typings/status.py`와 동일한 값

**바뀐 것은 딱 하나, MySQL을 Docker 컨테이너 대신 Colab에 직접 설치한 서버로 씁니다** (기능은 동일).
출력 파일(`dbbench_std_runs_vicuna-13b-local.jsonl`)은 다른 4개 모델과 완전히 같은 포맷이라,
다운로드해서 `project_db/results/raw/`에 넣고 `evaluate.py`/`analyze.py`의 `MODELS` 리스트에
`"vicuna-13b-local"`만 추가하면 바로 5번째 모델로 합류합니다.

**300문제 전체는 오래 걸릴 수 있습니다** — 아래 `LIMIT` 변수로 먼저 5~10문제만 테스트해보고,
문제 없으면 `LIMIT = None`으로 바꿔서 전체를 돌리는 걸 추천합니다. 중간에 끊겨도 이미 처리한
인덱스는 건너뛰고 이어서 진행합니다(재실행 시 자동 resume).

In [ ]:
# 저장소 클론 (project_db/data/dbbench_standard_full.jsonl이 필요합니다)
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    REPO_URL = "https://github.com/gksmfly/Agentbench-Mini-Eval.git"
    if not os.path.exists("Agentbench-Mini-Eval"):
        !git clone {REPO_URL}
    %cd Agentbench-Mini-Eval/project_db/notebooks

In [ ]:
# MySQL을 Docker 없이 직접 설치 (Colab은 보통 Docker를 지원하지 않음)
!apt-get update -qq
!apt-get install -y -qq mysql-server > /dev/null
!service mysql start
!mysql -e "ALTER USER 'root'@'localhost' IDENTIFIED WITH mysql_native_password BY 'password'; FLUSH PRIVILEGES;"
print("MySQL 준비 완료")

In [ ]:
!pip install -q mysql-connector-python transformers accelerate bitsandbytes

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(없음, CPU로 진행 시 매우 느릴 수 있음)")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "lmsys/vicuna-13b-v1.5"  # 게이트 모델 아님 -- HF 토큰 불필요
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
print("모델 로드 완료")

# vicuna-13b-v1.5는 tokenizer_config.json에 chat_template이 없어서 apply_chat_template()이
# 에러를 냅니다 -- FastChat 공식 vicuna_v1.1 템플릿으로 프롬프트를 직접 조립합니다.
VICUNA_SYSTEM = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)

def build_vicuna_prompt(history):
    prompt = VICUNA_SYSTEM + " "
    for m in history:
        role = "USER" if m["role"] == "user" else "ASSISTANT"
        prompt += f"{role}: {m['content']}" + (" " if role == "USER" else "</s>")
    return prompt + "ASSISTANT:"

def generate_vicuna(history: list[dict], max_new_tokens: int = 512) -> str:
    prompt = build_vicuna_prompt(history)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

In [ ]:
import re
import mysql.connector

# THUDM/AgentBench src/server/tasks/dbbench/__init__.py의 big_prompt 원문 그대로
SYSTEM_PROMPT = """
I will ask you a question, then you should help me operate a MySQL database with SQL to answer the question.
You have to explain the problem and your solution to me and write down your thoughts.
After thinking and explaining thoroughly, every round you can choose to operate or to answer.
your operation should be like this:
Action: Operation
```sql
SELECT * FROM table WHERE condition;
```
You MUST put SQL in markdown format without any other comments. Your SQL should be in one line.
Every time you can only execute one SQL statement. I will only execute the statement in the first SQL code block. Every time you write a SQL, I will execute it for you and give you the output.
If you are done operating, and you want to commit your final answer, then write down:
Action: Answer
Final Answer: ["ANSWER1", "ANSWER2", ...]
DO NOT write this pattern unless you are sure about your answer. I expect an accurate and correct answer.
Your answer should be accurate. Your answer must be exactly the same as the correct answer.
If the question is about modifying the database, then after done operation, your answer field can be anything.
If your response cannot match any pattern I mentioned earlier, you will be judged as FAIL immediately.
Your input will be raw MySQL response, you have to deal with it by yourself.
"""


# AgentBench/src/server/tasks/dbbench/__init__.py의 build_init_sql() 그대로 이식
def build_init_sql(entry):
    name = entry["table"]["table_name"]
    columns = ",".join([f"`{c['name']}` TEXT" for c in entry["table"]["table_info"]["columns"]])
    column_names = ",".join([f"`{c['name']}`" for c in entry["table"]["table_info"]["columns"]])
    items = []
    items_data = ()
    for row in entry["table"]["table_info"]["rows"]:
        item = "("
        for col in row:
            item += "%s,"
            items_data += (col,)
        item = item[:-1] + ")"
        items.append(item)
    items = ",".join(items)
    sql = f"""CREATE DATABASE IF NOT EXISTS `{name}`;
USE `{name}`;
CREATE TABLE IF NOT EXISTS `{name}` ({columns});
INSERT INTO `{name}` ({column_names}) VALUES {items};
COMMIT;
"""
    return sql, items_data


# AgentBench/src/server/tasks/dbbench/Interaction.py의 Container 그대로 이식,
# Docker 컨테이너 대신 Colab에 직접 설치한 MySQL(127.0.0.1:3306)에 연결
class Container:
    def __init__(self):
        self.conn = mysql.connector.connect(
            host="127.0.0.1", user="root", password="password", port=3306,
            pool_reset_session=True,
        )

    def execute(self, sql, database=None, data=()):
        self.conn.reconnect()
        try:
            with self.conn.cursor() as cursor:
                if database:
                    cursor.execute(f"use `{database}`;")
                    cursor.fetchall()
                cursor.execute(sql, data, multi=True)
                result = cursor.fetchall()
                result = str(result)
            self.conn.commit()
        except Exception as e:
            result = str(e)
        if len(result) > 800:
            result = result[:800] + "[TRUNCATED]"
        return result


print("채점 로직 준비 완료")

In [ ]:
# AgentBench/src/server/tasks/dbbench/__init__.py의 DBBench.start_sample() 로직 그대로 이식
# (session.action() 호출만 generate_vicuna()로 교체)
MAX_ROUND = 15

def run_sample(entry, container):
    db = entry["table"]["table_name"]
    init_sql, init_data = build_init_sql(entry)
    container.execute(init_sql, data=init_data)

    history = [
        {"role": "user", "content": SYSTEM_PROMPT},
        {"role": "assistant", "content": "Ok."},
    ]
    prompt = entry["description"] + "\n" + entry["add_description"]
    history.append({"role": "user", "content": prompt})

    res = generate_vicuna(history)
    history.append({"role": "assistant", "content": res})

    answer, error, finish_reason = "", "", "completed"
    try:
        action = re.search(r"Action: (.*?)\n", res)
        rounds = 0
        while action and action.group(1) == "Operation" and rounds < MAX_ROUND:
            sql_match = re.search(r"```sql\n([\s\S]*?)\n```", res)
            if not sql_match:
                finish_reason = "agent validation failed"
                break
            sql = sql_match.group(1).strip().replace("\n", " ")
            response = container.execute(sql, db)
            history.append({"role": "user", "content": response if response else ""})
            res = generate_vicuna(history)
            history.append({"role": "assistant", "content": res})
            action = re.search(r"Action: (.*?)\n", res)
            rounds += 1
        else:
            m = re.search(r"\nFinal Answer:(.*)", res)
            if m:
                answer = m.group(1)
            else:
                answer = ""
                finish_reason = "agent validation failed"
            if rounds >= MAX_ROUND and not answer:
                finish_reason = "task limit reached"
    except Exception as e:
        error = str(e)
        answer = ""
        finish_reason = "unknown"

    if entry["type"][0] in ("INSERT", "DELETE", "UPDATE"):
        columns = ",".join(f"`{c['name']}`" for c in entry["table"]["table_info"]["columns"])
        md5_query = (
            f"select md5(group_concat(rowhash order by rowhash)) as hash "
            f"from( SELECT substring(MD5(CONCAT_WS(',', {columns})), 1, 5) AS rowhash FROM `{db}`) as sub;"
        )
        answer = container.execute(md5_query, db)

    container.execute(f"drop database `{db}`")

    return {
        "status": finish_reason,
        "result": {"answer": str(answer), "type": entry["type"][0], "error": error},
        "history": history,
    }

print("run_sample() 준비 완료")

In [ ]:
import json
import time
from pathlib import Path

DATA_FILE = Path("../data/dbbench_standard_full.jsonl")
OUT_FILE = Path("../results/raw/dbbench_std_runs_vicuna-13b-local.jsonl")
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)

LIMIT = 10  # 먼저 10문제만 테스트해보고, 문제 없으면 None으로 바꿔서 전체 300문제 실행

entries = [json.loads(l) for l in open(DATA_FILE)]
if LIMIT is not None:
    entries = entries[:LIMIT]

# 이미 처리된 인덱스는 건너뜀 (중간에 끊겨도 재실행하면 이어서 진행)
done_indices = set()
if OUT_FILE.exists():
    with open(OUT_FILE) as f:
        for line in f:
            done_indices.add(json.loads(line)["index"])
print(f"전체 {len(entries)}문제 중 이미 완료된 {len(done_indices)}개는 건너뜁니다")

container = Container()
start_time = time.time()

with open(OUT_FILE, "a") as out:
    for index, entry in enumerate(entries):
        if index in done_indices:
            continue
        t0 = time.time()
        output = run_sample(entry, container)
        output["index"] = index
        elapsed = time.time() - t0
        record = {
            "index": index,
            "error": None,
            "info": None,
            "output": output,
            "time": {"timestamp": int(time.time() * 1000), "str": time.strftime("%Y-%m-%d %H:%M:%S")},
        }
        out.write(json.dumps(record) + "\n")
        out.flush()
        total_elapsed = time.time() - start_time
        print(f"[{index+1}/{len(entries)}] status={output['status']:<24} "
              f"{elapsed:5.1f}s (누적 {total_elapsed/60:5.1f}분)")

print(f"\n완료: {OUT_FILE}")

## 다음 단계

1. 위 셀에서 `LIMIT = 10`으로 먼저 테스트해보고 정상 동작(에러 없음, `status`가 매 문제 출력됨)을 확인하세요.
2. 확인되면 `LIMIT = None`으로 바꿔서 다시 실행 — 이미 처리된 인덱스는 자동으로 건너뛰고 이어서 진행합니다.
3. 300문제가 다 끝나면 `project_db/results/raw/dbbench_std_runs_vicuna-13b-local.jsonl`을 다운로드해서
   저장소의 같은 경로에 넣으세요.
4. `project_db/src/evaluate.py`와 `project_db/src/analyze.py`의 `MODELS` 리스트에
   `"vicuna-13b-local"`을 추가하고 두 스크립트를 다시 실행하면 5번째 모델로 결과가 합류합니다.